In [ ]:
# ==========================================
# CELL 1: SETUP API & PATH FILE (VERSI GEMINI)
# ==========================================
import os
import time
import pandas as pd
import google.generativeai as genai

# Setup API Key Gemini
# Ganti dengan key asli Anda jika os.getenv gagal: api_key_gemini = "AIzaSy..."
api_key_gemini = os.getenv("GEMINI_API_KEY")

if api_key_gemini:
    # Cek apakah terbaca (Print 8 karakter pertama saja untuk keamanan)
    print(f"🔑 Key yang terbaca oleh sistem: {api_key_gemini[:8]}...") 
    # Jika hasil print memunculkan 'AIza...' (tanpa tanda kutip), berarti SUKSES!
    
    # Konfigurasi Gemini API
    genai.configure(api_key=api_key_gemini)
    
    # Inisialisasi Model (Disiapkan untuk Cell 3 nanti)
    model = genai.GenerativeModel('gemini-2.5-flash')
    print("✅ Client Gemini 2.5 Flash berhasil diinisialisasi!")
else:
    print("❌ ERROR: API Key Gemini tidak ditemukan! Pastikan environment variable sudah diset.")

# Setup Path (Sesuaikan dengan struktur folder repo exigen-smart-maintenance Anda)
path_df_aset = "../../../data/master_aset_enriched.xlsx" 
path_df_perbaikan = "../../../data/aset_komplain_enriched.xlsx"
path_output = "../../../data/dataset_tiket_lengkap.csv"

print("✅ Cell 1 Selesai: Library ter-import dan Path sudah diset.")

🔑 Key yang terbaca oleh sistem: AIzaSyBh...
✅ Client Gemini 2.5 Flash berhasil diinisialisasi!
✅ Cell 1 Selesai: Library ter-import dan Path sudah diset.


d:\05_Personal\College\semester-6\NTG-Project\exigen-smart-maintenance\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\ryama\AppData\Local\Temp\ipykernel_24588\2119274788.py:7: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
# ==========================================
# CELL 2: LOAD DATA & AMBIL ATURAN (CONSTRAINTS)
# ==========================================
print("Membaca dataset asli NTG...")
df_aset = pd.read_excel(path_df_aset)
df_perbaikan = pd.read_excel(path_df_perbaikan)

# Mengambil daftar Kunci Jawaban (Target Y) agar AI tidak ngarang
list_severity = df_perbaikan['Severity'].dropna().unique().tolist()
list_penyebab = df_perbaikan['Penyebab'].dropna().unique().tolist()
list_jenis_kerusakan = df_perbaikan['Jenis Kerusakan'].dropna().unique().tolist() 

# Mengambil Kombinasi Aset Nyata (Feature X)
kombinasi_valid = df_aset[['Kategori', 'Sub Kategori', 'Tipe', 'Merek', 
                           'Lokasi Gedung', 'Lokasi Lantai', 'Lokasi Zona']].dropna().drop_duplicates()

# UNTUK TESTING: Kita ambil 3 kombinasi acak saja dulu agar prosesnya cuma hitungan detik
kombinasi_sample = kombinasi_valid.sample(3) 

print(f"✅ Cell 2 Selesai: Berhasil mengekstrak {len(kombinasi_sample)} kombinasi mesin untuk di-testing.")
print(f"🔸 Batas Severity yang diizinkan: {list_severity[:3]}...")

Membaca dataset asli NTG...
✅ Cell 2 Selesai: Berhasil mengekstrak 3 kombinasi mesin untuk di-testing.
🔸 Batas Severity yang diizinkan: ['Fatal', 'Ringan', 'Sedang']...


In [3]:
# ==========================================
# CELL 2.1: VALIDASI DATA UNIK (MAKESURE DATA BENAR)
# ==========================================

print("🔍 === RINGKASAN DATA UNIK UNTUK PROMPT AI === 🔍\n")

# 1. Validasi Kolom Target (dari df_perbaikan)
print("--- [TARGET Y / KUNCI JAWABAN] ---")
print(f"🔹 Severity ({len(list_severity)}): {list_severity}")
print(f"🔹 Penyebab ({len(list_penyebab)}): {list_penyebab[:10]}... (dan seterusnya)")
print(f"🔹 Tindakan ({len(list_jenis_kerusakan)}): {list_jenis_kerusakan[:10]}... (dan seterusnya)")

# 2. Validasi Identitas & Lokasi (dari df_aset)
print("\n--- [IDENTITAS ASET & LOKASI] ---")
cols_aset = ['Kategori', 'Sub Kategori', 'Tipe', 'Merek', 'Lokasi Gedung', 'Lokasi Lantai', 'Lokasi Zona']

for col in cols_aset:
    unique_vals = df_aset[col].dropna().unique().tolist()
    print(f"📍 {col} ({len(unique_vals)} unik): {unique_vals[:15]}") # Tampilkan 15 contoh pertama

# 3. Validasi Kombinasi yang Terpilih untuk Testing
print("\n--- [CONTOH KOMBINASI UNTUK TESTING GROQ] ---")
display(kombinasi_sample)

🔍 === RINGKASAN DATA UNIK UNTUK PROMPT AI === 🔍

--- [TARGET Y / KUNCI JAWABAN] ---
🔹 Severity (4): ['Fatal', 'Ringan', 'Sedang', 'Berat']
🔹 Penyebab (12): ['Kelembaban tinggi', 'Human error', 'Overload', 'Kurang perawatan', 'Tegangan tidak stabil', 'Faktor lingkungan', 'Debu/kotoran', 'Aus normal', 'Usia pakai', 'Kualitas material']... (dan seterusnya)
🔹 Tindakan (20): ['Retak/pecah', 'Overheat', 'Remote tidak berfungsi', 'Getaran berlebihan', 'Korsleting', 'Aus/abrasi', 'Tidak berfungsi total', 'Kebocoran', 'Sensor error', 'Aliran lemah']... (dan seterusnya)

--- [IDENTITAS ASET & LOKASI] ---
📍 Kategori (15 unik): ['Mechanical', 'Ventilasi Sistem', 'Electrical', 'Sistem Pemadam Kebakaran', 'Sistem Telekomunikasi Gedung', 'Sistem Proteksi Kebakaran Aktif', 'Security Sistem', 'Civil', 'Plumbing', 'Distribusi Air', 'Sistem Transportasi Gedung', 'Pencatatan Meter', 'Arsitektur', 'Sistem Energi', 'Latihan Balakar']
📍 Sub Kategori (51 unik): ['Tata Udara', 'Sistem Sirkulasi Udara', 'Contro

,Kategori,Sub Kategori,Tipe,Merek,Lokasi Gedung,Lokasi Lantai,Lokasi Zona
24271,Plumbing,Sanitari Sistem,Floor Drain,Generic,Gedung A,8,Tengah
38833,Civil,Dinding Bangunan,Dinding Tembok/Mansonry,Generic,Gedung C,7,Selatan
4385,Distribusi Air,Distributor Air Bersih,Ground Water Tank,Generic,Gedung E,6,Tengah


In [4]:
# ==========================================
# CELL 2.2: STRATIFIED SAMPLING (JAMINAN 100% COVERAGE)
# ==========================================

print("Membuat antrean kasus komprehensif agar tidak ada variasi yang terlewat...")

# 1. Ambil SEMUA variasi unik dari histori perbaikan (Dijamin 100% terwakili)
kasus_unik = df_perbaikan.drop_duplicates(
    subset=['Kategori', 'Severity', 'Penyebab', 'Jenis Kerusakan']
).copy()

print(f"Total variasi kasus unik di histori: {len(kasus_unik)} kombinasi.")

# 2. Pasangkan setiap kasus unik dengan spesifikasi Aset Nyata (Merek, Lokasi, dll)
antrean_prompt = []

for _, row in kasus_unik.iterrows():
    kategori_kasus = row['Kategori']
    
    # Cari aset di df_aset yang Kategori-nya cocok dengan kasus ini
    aset_cocok = kombinasi_valid[kombinasi_valid['Kategori'] == kategori_kasus]
    
    if not aset_cocok.empty:
        # Ambil 1 aset acak yang relevan untuk dijadikan "aktor" dalam skenario ini
        aset_pilih = aset_cocok.sample(1).iloc[0]
        
        antrean_prompt.append({
            'Kategori': kategori_kasus,
            'Tipe': aset_pilih['Tipe'],
            'Merek': aset_pilih['Merek'],
            'Lokasi Gedung': aset_pilih['Lokasi Gedung'],
            'Lokasi Lantai': aset_pilih['Lokasi Lantai'],
            'Lokasi Zona': aset_pilih['Lokasi Zona'],
            'Severity': row['Severity'],
            'Penyebab': row['Penyebab'],
            'Jenis Kerusakan': row['Jenis Kerusakan'],
            'Biaya Perbaikan': row['Biaya Perbaikan']
        })

df_antrean = pd.DataFrame(antrean_prompt)
print(f"✅ Antrean Siap! Total data yang HARUS di-generate AI: {len(df_antrean)} baris.")
display(df_antrean.head())

Membuat antrean kasus komprehensif agar tidak ada variasi yang terlewat...
Total variasi kasus unik di histori: 9000 kombinasi.
✅ Antrean Siap! Total data yang HARUS di-generate AI: 9000 baris.


,Kategori,Tipe,Merek,Lokasi Gedung,Lokasi Lantai,Lokasi Zona,Severity,Penyebab,Jenis Kerusakan,Biaya Perbaikan
0,Mechanical,FCU,Lokal,Gedung C,3,Timur,Fatal,Kelembaban tinggi,Retak/pecah,18771000
1,Security Sistem,Kamera CCTV,Bosch,Gedung Utama,14,Timur,Ringan,Human error,Overheat,356000
2,Electrical,Lampu Wall sign,Import,Gedung D,4,Barat,Sedang,Overload,Remote tidak berfungsi,1610000
3,Mechanical,AC Split,Panasonic,Gedung C,4,Selatan,Ringan,Kurang perawatan,Overheat,122000
4,Security Sistem,DVR CCTV,Honeywell,Gedung D,10,Selatan,Sedang,Tegangan tidak stabil,Getaran berlebihan,1760000


In [ ]:
# ==========================================
# CELL 3: GENERATE DATASET DENGAN FITUR "RESUME" (VERSI GEMINI)
# ==========================================
import time
import os
import pandas as pd
import google.generativeai as genai

# Pastikan path sama dengan Cell 1 dan Cell 4
path_output = "../../../data/dataset_tiket_lengkap.csv" 

# 1. BACA FILE LAMA UNTUK CEK YANG SUDAH SELESAI
kombinasi_selesai = set()
if os.path.exists(path_output):
    try:
        df_lama = pd.read_csv(path_output, sep='|', on_bad_lines='skip')
        for _, row in df_lama.iterrows():
            # Gunakan try-except untuk mencegah error jika file lama headernya belum update
            try:
                kunci = f"{row['kategori_aset']}_{row['severity']}_{row['root_cause']}_{row['jenis_kerusakan']}"
                kombinasi_selesai.add(kunci)
            except KeyError:
                 pass
        print(f"✅ Menemukan file lama. {len(df_lama)} riwayat kasus dipulihkan.")
    except Exception as e:
        print(f"⚠️ Gagal membaca file lama, mulai dari awal. Error: {e}")

# 2. FILTER ANTREAN (Hanya ambil yang belum ada di file CSV)
antrean_sisa = []
for index, row in df_antrean.iterrows():
    kunci_antrean = f"{row['Kategori']}_{row['Severity']}_{row['Penyebab']}_{row['Jenis Kerusakan']}"
    if kunci_antrean not in kombinasi_selesai:
        antrean_sisa.append((index, row))

print(f"🎯 Total variasi unik keseluruhan: {len(df_antrean)}")
print(f"⏩ Sisa yang belum dikerjakan AI: {len(antrean_sisa)}\n")

# 3. PROSES LOOPING API GEMINI
csv_batch_baru = "" # Variabel penampung hasil baru

if len(antrean_sisa) > 0:
    print(f"🚀 Mulai memanggil Gemini untuk sisa antrean...")
    for index_asli, row in antrean_sisa:
        kat, tipe, merek = row['Kategori'], row['Tipe'], row['Merek']
        ged, lan, zon = row['Lokasi Gedung'], row['Lokasi Lantai'], row['Lokasi Zona']
        sev, rc, jk, biaya = row['Severity'], row['Penyebab'], row['Jenis Kerusakan'], row['Biaya Perbaikan']
        
        print(f"🔄 Memproses [Antrean {index_asli+1}/{len(df_antrean)}]: {kat} - {jk}...")
        
        # PROMPT GEMINI (11 Kolom Lengkap)
        # PROMPT GEMINI REVISI (PENAJAMAN SEVERITY)
        prompt = f"""
        Anda adalah AI pembuat dataset Machine Learning. TUGAS ANDA HANYA MENGELUARKAN TEKS CSV MURNI TANPA HEADER. 
        
        Buatkan 3 baris data CSV (3 variasi kalimat berbeda) untuk SATU skenario spesifik ini:
        - Tipe Aset: {tipe}
        - Kategori: {kat}
        - Merek: {merek}
        - Lokasi: Gedung {ged}, Lantai {lan}, Zona {zon}
        - Severity: {sev}
        - Root Cause: {rc}
        - Jenis Kerusakan: {jk}
        - Biaya Perbaikan: {biaya}

        ATURAN KETAT UNTUK KOSAKATA KELUHAN (WAJIB DIIKUTI BERDASARKAN SEVERITY):
        - Jika Severity "Ringan": Gunakan kata-kata sepele (misal: sedikit, agak, kurang, kotor, berdebu).
        - Jika Severity "Sedang": Gunakan kata-kata mengganggu (misal: bocor, macet, berisik, netes).
        - Jika Severity "Berat": Gunakan kata-kata rusak (misal: mati, patah, tidak fungsi, jebol).
        - Jika Severity "Fatal": Gunakan kata-kata gawat darurat dan panik (misal: meledak, kebakaran, hancur, bahaya, nyangkut, korslet).

        Format WAJIB: CSV murni dengan pemisah '|'.
        Kolom: teks_keluhan_awam|teks_laporan_teknisi|tipe_aset|lokasi_gedung|lokasi_lantai|lokasi_zona|kategori_aset|severity|root_cause|jenis_kerusakan|biaya_perbaikan
        """
        
        try:
            # ⬇️ PERUBAHAN: Sintaks khusus pemanggilan Gemini API
            response = model.generate_content(
                prompt,
                generation_config=genai.types.GenerationConfig(
                    temperature=0.8, # Suhu tinggi agar variasi bahasa WhatsApp lebih banyak
                    max_output_tokens=2000,
                )
            )
            
            # ⬇️ PERUBAHAN: Cara membaca output text dari Gemini
            hasil_text = response.text.strip() 
            
            # Bersihkan markdown jika masih ada
            hasil_text = hasil_text.replace("```csv", "").replace("```", "").strip()
            
            # Hapus header jika AI tidak sengaja mengeluarkannya
            if hasil_text.lower().startswith("teks_keluhan"):
                hasil_text = "\n".join(hasil_text.split("\n")[1:])
                
            csv_batch_baru += hasil_text + "\n" # Tampung di memori
            
            # ⬇️ PERUBAHAN: Jeda disesuaikan untuk Gemini Free Tier (15 request/menit)
            time.sleep(5) 
            
        except Exception as e:
            print(f"❌ Terkena Limit/Error pada baris {index_asli+1}. Looping dihentikan aman. Error: {e}")
            break # Berhenti, tapi csv_batch_baru TIDAK HILANG

    print("✅ Cell 3 Selesai! Silakan jalankan Cell 4 untuk menyimpan data yang baru di-generate.")
else:
    print("🎉 Hore! Semua variasi kasus sudah selesai di-generate. Tidak perlu memanggil API lagi.")

✅ Menemukan file lama. 669 riwayat kasus dipulihkan.
🎯 Total variasi unik keseluruhan: 9000
⏩ Sisa yang belum dikerjakan AI: 8780

🚀 Mulai memanggil Gemini untuk sisa antrean...
🔄 Memproses [Antrean 205/9000]: Mechanical - Sensor error...
🔄 Memproses [Antrean 206/9000]: Mechanical - Tidak berfungsi total...
❌ Terkena Limit/Error pada baris 206. Looping dihentikan aman. Error: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash
Please retry in 18.142790578s. [links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_

In [6]:
# ==========================================
# CELL 4: SIMPAN & BACA HASIL (APPEND MODE)
# ==========================================

# 1. Simpan Data Baru (Jika ada)
if 'csv_batch_baru' in locals() and csv_batch_baru.strip() != "":
    
    # Jika file belum pernah ada sama sekali, buatkan sekalian beserta headernya
    if not os.path.exists(path_output):
        with open(path_output, "w", encoding="utf-8") as f:
            # ⬇️ PERUBAHAN: Header disesuaikan menjadi 11 Kolom
            f.write("teks_keluhan_awam|teks_laporan_teknisi|tipe_aset|lokasi_gedung|lokasi_lantai|lokasi_zona|kategori_aset|severity|root_cause|jenis_kerusakan|biaya_perbaikan\n")
            
    # Tambahkan (Append) data baru ke baris paling bawah file CSV
    with open(path_output, "a", encoding="utf-8") as f:
        f.write(csv_batch_baru)
        
    print(f"💾 File berhasil di-update dan disimpan di: {path_output}")
    
    # KOSONGKAN variabel setelah disimpan agar tidak ter-save ganda jika Anda me-run Cell 4 dua kali
    csv_batch_baru = "" 
else:
    print("ℹ️ Tidak ada teks baru untuk ditambahkan ke file saat ini.")

# 2. Coba baca keseluruhan dataset menggunakan Pandas
try:
    df_hasil = pd.read_csv(path_output, sep='|', on_bad_lines='skip')
    print(f"\n📊 DATASET FINAL! Total Keseluruhan Data Saat Ini: {df_hasil.shape[0]} baris.")
    display(df_hasil.head())
    display(df_hasil.tail()) # Tampilkan juga bagian paling bawah untuk ngecek
except Exception as e:
    print(f"❌ Gagal membaca CSV. Error: {e}")

💾 File berhasil di-update dan disimpan di: ../../data/dataset_tiket_lengkap.csv

📊 DATASET FINAL! Total Keseluruhan Data Saat Ini: 672 baris.


,teks_keluhan_awam,teks_laporan_teknisi,tipe_aset,lokasi_gedung,lokasi_lantai,lokasi_zona,kategori_aset,severity,root_cause,jenis_kerusakan,biaya_perbaikan
0,AC Split di lantai 11 zona selatan gedung utam...,Laporan kerusakan AC Split Samsung di Gedung U...,AC Split,Gedung Utama,Lantai 11,Zona Selatan,Mechanical,Fatal,Kelembaban tinggi,Retak/pecah,18771000.0
1,AC Split di gedung utama lantai 11 zona selata...,Laporan inspeksi menunjukkan bahwa AC Split Sa...,AC Split,Gedung Utama,Lantai 11,Zona Selatan,Mechanical,Fatal,Kelembaban tinggi,Retak/pecah,18771000.0
2,Lagi ada masalah dengan AC Split di zona selat...,Tindakan perbaikan darurat diperlukan untuk AC...,AC Split,Gedung Utama,Lantai 11,Zona Selatan,Mechanical,Fatal,Kelembaban tinggi,Retak/pecah,18771000.0
3,Monitor CCTV di lantai 7 zona selatan gedung u...,Laporan perbaikan Monitor CCTV Samsung di Gedu...,Monitor CCTV,Gedung Utama,Lantai 7,Zona Selatan,Security Sistem,Ringan,Human error,Overheat,356000.0
4,CCTV samsung di gedung utama lantai 7 zona sel...,"Pemeriksaan CCTV Samsung di Gedung Utama, Lant...",Monitor CCTV,Gedung Utama,Lantai 7,Zona Selatan,Security Sistem,Ringan,Human error,Overheat,356000.0


,teks_keluhan_awam,teks_laporan_teknisi,tipe_aset,lokasi_gedung,lokasi_lantai,lokasi_zona,kategori_aset,severity,root_cause,jenis_kerusakan,biaya_perbaikan
667,"Halo, ada bunyi aneh dari Roof Tank di Lantai ...",Laporan teknisi: Roof Tank di Gedung Parkir La...,Roof Tank,Gedung Parkir,Lantai 1,Zona Selatan,Distribusi Air,Sedang,Overload,Bunyi abnormal,1250000.0
668,Roof Tank di Gedung Parkir Lantai 1 Zona Selat...,Laporan teknisi: Roof Tank di Gedung Parkir La...,Roof Tank,Gedung Parkir,Lantai 1,Zona Selatan,Distribusi Air,Sedang,Overload,Bunyi abnormal,1250000.0
669,"Mas, AC di Gedung C lantai 7 zona barat kok er...",Dilakukan inspeksi pada unit AC Cassette di Ge...,AC Cassette,Gedung C,Lantai 7,Zona Barat,Mechanical,Sedang,Getaran,Sensor error,1529000.0
670,HELP! AC Cassette di lantai 7 Gedung C zona ba...,Perbaikan unit AC Cassette merek Mitsubishi di...,AC Cassette,Gedung C,Lantai 7,Zona Barat,Mechanical,Sedang,Getaran,Sensor error,1529000.0
671,"Pak, AC di Gedung C, Lantai 7, Zona Barat, ini...","Penanganan keluhan AC Cassette di Gedung C, La...",AC Cassette,Gedung C,Lantai 7,Zona Barat,Mechanical,Sedang,Getaran,Sensor error,1529000.0
